# Rebuild every table and figure of the revision from the saved outputs

This notebook needs **no PyTorch and no re-training**. It only reads the files that Cell 14 of the main notebook saved, so you can run it in a fresh Colab session (also after a disconnect).

**Before you start:** upload to Colab (left panel > Files) `CANCER_DATA.csv` and `ngnn_revision_outputs.zip` (the zip from Cell 14). Run the cells in order. Everything is written to the folder `tables_figures/` and zipped at the end.

| Cell | Produces | Manuscript item |
| --- | --- | --- |
| 2 | Table 1, Table 2 | Sections 6.4, 6.5 |
| 3 | Table 3, Table 4, Figure 5 (reliability diagram) | Sections 6.6, 6.7 |
| 4 | Table 5 (bootstrap CIs) | Section 6.8 |
| 5 | Table 6 (DeLong, McNemar) | Section 6.9 |
| 6 | Table 7, decision curve | Section 6.10 |
| 7 | Figure 6 (confusion matrix), Table 8 | Sections 6.11, 6.12 |
| 8 | Table 9, Figure 7 (ablation) | Section 6.13 |
| 9 | One Excel file with all tables | for pasting into Word |

**Not rebuilt here (no data in them):** Figures 1 to 4 (workflow, membership sketch, architecture, GNN family). **Heatmap (Section 6.14):** already saved as `heatmap_aggregated_triplet.png` in your zip.

All out-of-fold tables use **repetition 0** (one prediction per patient), exactly as the manuscript did, and they use the **full NGNN-MADM**, not the simplified reference model.

In [ ]:
# Cell 1: load the saved outputs
import numpy as np, pandas as pd, os, zipfile, warnings; warnings.filterwarnings('ignore')
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

CSV, ZIP, OUT = '/content/CANCER_DATA.csv', '/content/ngnn_revision_outputs.zip', 'saved_outputs'
if not os.path.exists(f'{OUT}/oof_predictions.npz'): zipfile.ZipFile(ZIP).extractall(OUT)
FIG = 'tables_figures'; os.makedirs(FIG, exist_ok=True)

df = pd.read_csv(CSV); y = (df['diagnosis'] == 'M').astype(int).values; N = len(y)
z = np.load(f'{OUT}/oof_predictions.npz', allow_pickle=True)
Iagg = z['Iagg']; oof = {k: z[k] for k in z.files if k not in ('Iagg', 'T_final', 'I_final', 'F_final')}
fold_df = pd.read_csv(f'{OUT}/per_fold_metrics.csv')
key = lambda name: name.replace('-', '_').replace(' ', '_')
NAMES = {'NGNN_MADM': 'NGNN-MADM', 'RefNGNN_MADM': 'NGNN-MADM (ref.)', 'LR': 'LR', 'SVM': 'SVM', 'RF': 'RF', 'MLP': 'MLP',
         'LightGBM': 'LightGBM', 'XGBoost': 'XGBoost', 'GCN': 'GCN', 'GraphSAGE': 'GraphSAGE', 'CatBoost': 'CatBoost',
         'GAT': 'GAT', 'GIN': 'GIN', 'Graph_Transformer': 'Graph Transformer'}
MAIN = [k for k in NAMES if k in oof]                       # models that exist in the saved file
TARGET = 'NGNN_MADM' if 'NGNN_MADM' in oof else 'RefNGNN_MADM'
P0 = {k: oof[k][0] for k in oof}                              # repetition 0: one out-of-fold prediction per patient
print('models found:', MAIN); print('target:', TARGET, '| patients:', N, '| repetitions:', oof[TARGET].shape[0])
tables = {}                                                    # name -> DataFrame, written to Excel in the last cell

In [ ]:
# Cell 2: Table 1 and Table 2 (per-fold values, Nadeau-Bengio correction, Holm)
fm = fold_df[fold_df.model.map(key).isin(MAIN)].copy(); fm['k'] = fm.model.map(key)
kr, ratio = fm.groupby('k').size().max(), 0.25; tcrit = stats.t.ppf(0.975, kr - 1); rows = []
for k, g in fm.groupby('k'):
    sd = g.acc.std(ddof=1); half = tcrit * sd * np.sqrt(1/kr + ratio)
    rows.append({'Model': NAMES[k], 'Acc (%)': f'{g.acc.mean():.2f} +/- {sd:.2f}', '95% CI (Acc)': f'[{g.acc.mean()-half:.1f}, {g.acc.mean()+half:.1f}]',
                 'F1 (%)': round(g.f1.mean(), 2), 'AUC': round(g.auc.mean(), 3), 'Brier': round(g.brier.mean(), 3), '_s': g.acc.mean()})
t1 = pd.DataFrame(rows).sort_values('_s', ascending=False).drop(columns='_s'); tables['Table 1'] = t1; print(t1.to_string(index=False))

piv = fm.pivot(index='fold', columns='k', values='acc'); res = []
for k in [m for m in piv.columns if m != TARGET]:
    d = piv[TARGET] - piv[k]; sd = d.std(ddof=1); t = d.mean() / (sd * np.sqrt(1/kr + ratio))
    res.append({'vs': NAMES[k], 'Mean diff': d.mean(), 'SD of diff': sd, 'dz': d.mean()/sd, 't (NB)': t, 'p (raw)': 2*stats.t.sf(abs(t), kr-1)})
t2 = pd.DataFrame(res); o = t2['p (raw)'].argsort().values; adj = np.zeros(len(t2)); run = 0
for r, i in enumerate(o): run = max(run, (len(t2) - r) * t2['p (raw)'].iloc[i]); adj[i] = min(1, run)
t2['p (Holm)'] = adj; tables['Table 2'] = t2.round(3); print('\n', t2.round(3).to_string(index=False))

In [ ]:
# Cell 3: Table 3 (calibration), Table 4 (tests + paired bootstrap), Figure 5 (reliability diagram)
rng = np.random.default_rng(0)
def calib(p, yy=None):
    yy = y if yy is None else yy; bins = np.linspace(0, 1, 11); ece = mce = rel = res = 0.0; base = yy.mean()
    for lo, hi in zip(bins[:-1], bins[1:]):
        s = (p >= lo) & (p < hi + (hi == 1))
        if s.any():
            gap = abs(yy[s].mean() - p[s].mean()); w = s.mean(); ece += w*gap; mce = max(mce, gap)
            rel += w*(yy[s].mean() - p[s].mean())**2; res += w*(yy[s].mean() - base)**2
    lg = np.log(np.clip(p, 1e-6, 1-1e-6)/(1-np.clip(p, 1e-6, 1-1e-6))); cs = LogisticRegression(C=1e6, max_iter=1000).fit(lg[:, None], yy)
    return dict(ECE=ece, MCE=mce, Slope=cs.coef_[0, 0], Intercept=cs.intercept_[0], Brier=np.mean((p - yy)**2), Rel=rel, Res=res)
def spiegelhalter(p):
    zs = np.sum((y - p)*(1 - 2*p)) / np.sqrt(np.sum((1 - 2*p)**2 * p*(1 - p))); return zs, 2*stats.norm.sf(abs(zs))
T3 = ['LR', 'RF', TARGET] if all(m in P0 for m in ('LR', 'RF')) else [TARGET]
rows = []
for k in T3:
    c = calib(P0[k]); rows.append({'Model': NAMES[k], 'ECE': round(c['ECE'], 3), 'MCE': round(c['MCE'], 3), 'Slope': round(c['Slope'], 2),
                                   'Intercept': round(c['Intercept'], 2), 'Brier': round(c['Brier'], 4), 'Reliability / Resolution': f"{c['Rel']:.4f} / {c['Res']:.4f}"})
tables['Table 3'] = pd.DataFrame(rows); print(tables['Table 3'].to_string(index=False))

B = 2000; idx = rng.integers(0, N, (B, N)); rows = []
for k in T3:
    z_, p_ = spiegelhalter(P0[k]); row = {'Model': NAMES[k], 'Spiegelhalter z (p)': f'{z_:.2f} ({p_:.3f})'}
    if k != TARGET:
        db, de = [], []
        for b in idx:
            a, c = calib(P0[k][b], y[b]), calib(P0[TARGET][b], y[b]); db.append(a['Brier'] - c['Brier']); de.append(a['ECE'] - c['ECE'])
        f = lambda v, fmt: f'{np.mean(v):+{fmt}} [{np.percentile(v, 2.5):{fmt}}, {np.percentile(v, 97.5):{fmt}}]'
        row['dBrier vs NGNN'] = f(db, '.4f'); row['dECE vs NGNN'] = f(de, '.3f')
    else: row['dBrier vs NGNN'] = row['dECE vs NGNN'] = '-'
    rows.append(row)
tables['Table 4'] = pd.DataFrame(rows); print(tables['Table 4'].to_string(index=False))

plt.figure(figsize=(5.5, 5.5)); bins = np.linspace(0, 1, 11); plt.plot([0, 1], [0, 1], 'k--', lw=.8, label='Perfect')
for k in T3:
    xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        s = (P0[k] >= lo) & (P0[k] < hi + (hi == 1))
        if s.sum() > 0: xs.append(P0[k][s].mean()); ys.append(y[s].mean())
    plt.plot(xs, ys, 'o-', label=NAMES[k])
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed fraction malignant'); plt.legend(); plt.tight_layout()
plt.savefig(f'{FIG}/Fig5_reliability.png', dpi=200); plt.show()

In [ ]:
# Cell 4: Table 5 (accuracy, sensitivity, specificity, precision, F1, AUC with bootstrap 95% CI)
def metrics(p, yy):
    pr = p >= .5; tp = (pr & (yy == 1)).sum(); tn = (~pr & (yy == 0)).sum(); fp = (pr & (yy == 0)).sum(); fn = (~pr & (yy == 1)).sum()
    prec = tp/(tp+fp) if tp+fp else np.nan; sens = tp/(tp+fn)
    return {'Accuracy (%)': 100*(tp+tn)/len(yy), 'Sensitivity (%)': 100*sens, 'Specificity (%)': 100*tn/(tn+fp), 'Precision (%)': 100*prec,
            'F1 (%)': 100*2*prec*sens/(prec+sens), 'AUC': roc_auc_score(yy, p)}
cols = {}
for k in T3:
    base = metrics(P0[k], y); boot = pd.DataFrame([metrics(P0[k][b], y[b]) for b in idx[:1000] if len(np.unique(y[b])) == 2])
    cols[NAMES[k]] = {m: (f'{base[m]:.3f} [{boot[m].quantile(.025):.3f}, {boot[m].quantile(.975):.3f}]' if m == 'AUC'
                          else f'{base[m]:.1f} [{boot[m].quantile(.025):.1f}, {boot[m].quantile(.975):.1f}]') for m in base}
tables['Table 5'] = pd.DataFrame(cols); print(tables['Table 5'].to_string())
print('Accuracy of', NAMES[TARGET], 'per repetition (%):', [round(100*np.mean((oof[TARGET][r] >= .5) == y), 1) for r in range(oof[TARGET].shape[0])], '-> Table 5 uses repetition 0 only; say so in the caption')

In [ ]:
# Cell 5: Table 6 (DeLong and exact McNemar), repetition 0 and median over the 10 repetitions
def midrank(x):
    o = np.argsort(x); s = x[o]; n = len(x); r = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and s[j] == s[i]: j += 1
        r[i:j] = 0.5*(i + j - 1) + 1; i = j
    out = np.empty(n); out[o] = r; return out
def delong(yy, p1, p2):
    pos = yy == 1; m, n = pos.sum(), (~pos).sum(); V10, V01, a = [], [], []
    for p in (p1, p2):
        tx, ty, tz = midrank(p[pos]), midrank(p[~pos]), midrank(p)
        a.append((tz[pos].sum() - m*(m+1)/2)/(m*n)); V10.append((tz[pos] - tx)/n); V01.append(1 - (tz[~pos] - ty)/m)
    S = np.cov(np.array(V10))/m + np.cov(np.array(V01))/n; var = S[0, 0] + S[1, 1] - 2*S[0, 1]
    zz = (a[0] - a[1])/np.sqrt(var); return a[0] - a[1], zz, 2*stats.norm.sf(abs(zz))
def mcnemar(yy, p1, p2):
    c1, c2 = (p1 >= .5) == yy, (p2 >= .5) == yy; b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    return b, c, (1.0 if b + c == 0 else min(1.0, 2*stats.binom.cdf(min(b, c), b + c, .5)))
rows = []
for k in [m for m in MAIN if m != TARGET]:
    dA, zz, pD = delong(y, oof[TARGET][0], oof[k][0]); b, c, pM = mcnemar(y, oof[TARGET][0], oof[k][0])
    pds, pms = [], []
    for r in range(oof[TARGET].shape[0]):
        pds.append(delong(y, oof[TARGET][r], oof[k][r])[2]); pms.append(mcnemar(y, oof[TARGET][r], oof[k][r])[2])
    rows.append({'Comparison': f'{NAMES[TARGET]} vs. {NAMES[k]}', 'dAUC': round(dA, 4), 'DeLong z (p), rep 0': f'{zz:.2f} ({pD:.3f})', 'b, c': f'{b}, {c}',
                 'McNemar exact p, rep 0': round(pM, 4), 'DeLong median p (10 reps)': round(np.median(pds), 3), 'McNemar median p (10 reps)': round(np.median(pms), 3)})
t6 = pd.DataFrame(rows); mp = t6['McNemar exact p, rep 0'].values; o = mp.argsort(); adj = np.zeros(len(mp)); run = 0
for r, i in enumerate(o): run = max(run, (len(mp) - r)*mp[i]); adj[i] = min(1, run)
t6['McNemar Holm (rep 0)'] = adj.round(3); tables['Table 6'] = t6; print(t6.to_string(index=False))

In [ ]:
# Cell 6: Table 7 (net benefit) and the full decision curve
def nb(p, t):
    pr = p >= t; return (pr & (y == 1)).sum()/N - (pr & (y == 0)).sum()/N * t/(1 - t)
prev = y.mean(); ths = [0.05, 0.1, 0.2, 0.3, 0.5]; rows = []
for k in [m for m in ['LR', 'SVM', 'RF', TARGET] if m in P0]: rows.append({'Strategy': NAMES[k], **{f'pt={t}': round(nb(P0[k], t), 3) for t in ths}})
rows.append({'Strategy': 'Treat all', **{f'pt={t}': round(prev - (1 - prev)*t/(1 - t), 3) for t in ths}})
rows.append({'Strategy': 'Treat none', **{f'pt={t}': 0.0 for t in ths}})
tables['Table 7'] = pd.DataFrame(rows); print(tables['Table 7'].to_string(index=False))
pts = np.linspace(0.01, 0.5, 50); plt.figure(figsize=(7, 4.5))
for k in [m for m in ['LR', 'SVM', 'RF', TARGET] if m in P0]: plt.plot(pts, [nb(P0[k], t) for t in pts], label=NAMES[k])
plt.plot(pts, [prev - (1 - prev)*t/(1 - t) for t in pts], '--', label='Treat all'); plt.axhline(0, color='k', lw=.8, label='Treat none')
plt.axvspan(0.05, 0.3, alpha=.08); plt.ylim(-0.1, 0.4); plt.xlabel('Threshold probability'); plt.ylabel('Net benefit'); plt.legend(); plt.tight_layout()
plt.savefig(f'{FIG}/Fig_decision_curve.png', dpi=200); plt.show()

In [ ]:
# Cell 7: Figure 6 (confusion matrix) and Table 8 (error and indeterminacy analysis, full NGNN-MADM, repetition 0)
p = P0[TARGET]; pr = p >= .5; conf = np.maximum(p, 1 - p); ia = Iagg[0]
cm = confusion_matrix(y, pr.astype(int)); plt.figure(figsize=(4, 3.6)); plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)
plt.xticks([0, 1], ['Benign', 'Malignant']); plt.yticks([0, 1], ['Benign', 'Malignant']); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
plt.savefig(f'{FIG}/Fig6_confusion.png', dpi=200); plt.show()
def comp(s): return f"{((pr == y) & s).sum()} correct, {((pr == 1) & (y == 0) & s).sum()} FP, {((pr == 0) & (y == 1) & s).sum()} FN"
grp = {'Correct, confidence >= 0.9': (pr == y) & (conf >= .9), 'Correct, confidence 0.7-0.9': (pr == y) & (conf >= .7) & (conf < .9),
       'Correct, confidence < 0.7': (pr == y) & (conf < .7), 'False positives': (pr == 1) & (y == 0), 'False negatives': (pr == 0) & (y == 1),
       'High indeterminacy (I_agg >= 0.5)': ia >= .5, 'Low indeterminacy (I_agg < 0.5)': ia < .5}
rows = [{'Group': g, 'n (% of 569)': f'{s.sum()} ({100*s.mean():.1f}%)', 'mean p': round(p[s].mean(), 2), 'mean I_agg': round(ia[s].mean(), 2), 'Composition': comp(s)} for g, s in grp.items()]
tables['Table 8'] = pd.DataFrame(rows); print(tables['Table 8'].to_string(index=False))
err = (pr != y).astype(int); a, b_, c_, d_ = int(err[ia >= .5].sum()), int((ia >= .5).sum() - err[ia >= .5].sum()), int(err[ia < .5].sum()), int((ia < .5).sum() - err[ia < .5].sum())
OR = (a*d_)/(b_*c_); se = np.sqrt(1/a + 1/b_ + 1/c_ + 1/d_)
print(f'Errors with I_agg>=0.5: {a}/{a+b_}; with I_agg<0.5: {c_}/{c_+d_}; OR {OR:.1f} [{np.exp(np.log(OR)-1.96*se):.1f}, {np.exp(np.log(OR)+1.96*se):.1f}], Fisher p = {stats.fisher_exact([[a, b_], [c_, d_]])[1]:.2g}')
print('AUROC error detection over repetitions: I_agg %.3f | 1-confidence %.3f' % (
    np.mean([roc_auc_score((oof[TARGET][r] >= .5) != y, Iagg[r]) for r in range(oof[TARGET].shape[0]) if ((oof[TARGET][r] >= .5) != y).sum() > 1]),
    np.mean([roc_auc_score((oof[TARGET][r] >= .5) != y, 1 - np.maximum(oof[TARGET][r], 1 - oof[TARGET][r])) for r in range(oof[TARGET].shape[0]) if ((oof[TARGET][r] >= .5) != y).sum() > 1])))

In [ ]:
# Cell 8: Table 9 and Figure 7 (ablation, from ablation_paired_tests.csv and per_fold_metrics.csv)
ab = pd.read_csv(f'{OUT}/ablation_paired_tests.csv')
full = fold_df[fold_df.model == 'NGNN-MADM'].acc; fm_, fs_ = full.mean(), full.std(ddof=1)
t9 = ab.rename(columns={'variant': 'Variant', 'acc': 'Accuracy (%)', 'diff_vs_full': 'Delta vs full (pp)', 'SD_d': 'SD of diff.', 'p_NB': 'p (corrected t)', 'p_Holm': 'p (Holm)'}).round(3)
tables['Table 9'] = t9; print(f'Full model: {fm_:.2f} +/- {fs_:.2f}'); print(t9.to_string(index=False))
means = [fold_df[fold_df.model == v].acc.mean() for v in ab.variant]; sds = [fold_df[fold_df.model == v].acc.std(ddof=1) for v in ab.variant]
plt.figure(figsize=(8, 4.5)); plt.axhspan(fm_ - fs_, fm_ + fs_, alpha=.15, color='C0', label='Full model mean +/- SD'); plt.axhline(fm_, color='C0')
plt.errorbar(range(len(means)), means, yerr=sds, fmt='o', capsize=3, color='C1'); plt.xticks(range(len(means)), ab.variant, rotation=40, ha='right')
plt.ylabel('Accuracy (%)'); plt.legend(); plt.tight_layout(); plt.savefig(f'{FIG}/Fig7_ablation.png', dpi=200); plt.show()

In [ ]:
# Cell 9: save everything: one Excel file (all tables), one CSV per table, and a zip
with pd.ExcelWriter(f'{FIG}/all_tables.xlsx') as xw:
    for name, t in tables.items():
        t.to_excel(xw, sheet_name=name, index=False); t.to_csv(f'{FIG}/{name.replace(" ", "_")}.csv', index=False)
import shutil; shutil.make_archive('tables_figures', 'zip', FIG); print(sorted(os.listdir(FIG)))
try:
    from google.colab import files; files.download('tables_figures.zip')
except Exception: pass